# E3 — Entrenamiento PCN v4

**Cambios respecto a v3:**
- **Datos:** `sintetico_roturas_v2` (2.299 pares, modos plano+chip+cuña) en lugar de `sintetico_roturas` (2.367 pares, solo plano)
- **Filtro outliers activo** en `dataset.py` (σ=2.5, elimina artefactos flotantes en el 12.4% de pares)
- **`w_coarse 0.5 → 1.0`**: peso igual para decoder coarse y fine → la forma global aprende mejor antes de refinar
- **`epochs 400 → 500`**: v3 convergió en época 347, se da más margen
- Todo lo demás igual a v3: LR=1e-4, decay cada 100, batch=64

**Resultado v3 de referencia:** CD=0.0665, F-Score=0.024 (época 347 de 400)

⏱️ **Tiempo estimado en A100: ~2.5 horas | en T4: ~5 horas**

---
### Antes de ejecutar:
Menú → **Entorno de ejecución → Cambiar tipo → A100 GPU** (o T4 si no hay A100 disponible)

In [ ]:
# ── CELDA 1: Montar Drive ──────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── CELDA 2: Clonar repo e instalar dependencias ───────────────
import os
from getpass import getpass

REPO_DIR = '/content/TFM'

if not os.path.exists(REPO_DIR):
    token = getpass('Pega tu token de GitHub (ghp_...) y pulsa Enter: ')
    repo_url = f'https://{token}@github.com/herredoble/TFM-reconstruccion-3D'
    os.system(f'git clone {repo_url} {REPO_DIR}')
    del token
else:
    os.system(f'git -C {REPO_DIR} pull')

os.chdir(REPO_DIR)
os.system('git checkout raquel/e3')
print('Directorio de trabajo:', os.getcwd())

import subprocess
subprocess.run(['pip', 'install', 'torch', 'numpy', 'matplotlib', '--quiet'])
print('Dependencias instaladas.')

In [ ]:
# ── CELDA 3: RUTAS DE DRIVE ────────────────────────────────────
DRIVE   = '/content/drive/MyDrive'
BASE_E3 = f'{DRIVE}/Datos_E2_E3/E3/Raquel'

VERSION = 'v4_pcn'

# ── NUEVO en v4: apunta a sintetico_roturas_v2 ────────────────
RUTA_SINTETICO    = f'{DRIVE}/Datos_E2_E3/General/sintetico_roturas_v2'
RUTA_FB_PROCESADO = f'{DRIVE}/Datos_E2_E3/General/Fantastik_Break_Preprocesado'

# Modelo v3 como referencia (para comparar métricas al final)
RUTA_MODELO_V3 = f'{BASE_E3}/modelos/v3_pcn/best.pt'

RUTA_SALIDA_MODELO     = f'{BASE_E3}/modelos/{VERSION}'
RUTA_SALIDA_RESULTADOS = f'{BASE_E3}/resultados/{VERSION}'

print('Rutas v4:')
print(f'  sintetico_v2    : {RUTA_SINTETICO}')
print(f'  fantastic_breaks: {RUTA_FB_PROCESADO}')
print(f'  modelo v3 (ref) : {RUTA_MODELO_V3}')
print(f'  salida modelo   : {RUTA_SALIDA_MODELO}')
print(f'  salida resultados: {RUTA_SALIDA_RESULTADOS}')

In [ ]:
# ── CELDA 4: Verificar que las rutas existen ───────────────────
from pathlib import Path

rutas = {
    'sintetico_roturas_v2'   : RUTA_SINTETICO,
    'fantastic_breaks'       : RUTA_FB_PROCESADO,
    'modelo_v3 (referencia)' : RUTA_MODELO_V3,
}

for nombre, ruta in rutas.items():
    existe = Path(ruta).exists()
    emoji  = '✅' if existe else '⚠️ '
    print(f'  {emoji} {nombre}: {ruta}')

# Contar pares disponibles
n_sint = len(list(Path(RUTA_SINTETICO).glob('*_completo.npy'))) if Path(RUTA_SINTETICO).exists() else 0
n_fb   = len(list(Path(RUTA_FB_PROCESADO).glob('*_completo.npy'))) if Path(RUTA_FB_PROCESADO).exists() else 0
print(f'\nPares disponibles:')
print(f'  sintetico_v2    : {n_sint}')
print(f'  fantastic_breaks: {n_fb}')
print(f'  TOTAL           : {n_sint + n_fb}')

In [ ]:
# ── CELDA 5: Copiar datos desde Drive ─────────────────────────
import subprocess
from pathlib import Path

def copiar_dir(src, dst):
    src, dst = Path(src), Path(dst)
    if dst.exists() and any(dst.glob('*.npy')):
        n = len(list(dst.glob('*.npy')))
        print(f'  [OK] ya existe: {dst.name}  ({n} .npy)')
        return
    if not src.exists():
        print(f'  [ERROR] no encontrado: {src}')
        return
    dst.mkdir(parents=True, exist_ok=True)
    print(f'  Copiando {src.name} (puede tardar varios minutos)...', flush=True)
    r = subprocess.run(['rsync', '-a', '--no-links', f'{src}/', str(dst)],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print(f'  [ERROR rsync] {r.stderr[:300]}')
    else:
        n = len(list(dst.glob('*.npy')))
        print(f'  listo — {n} archivos .npy copiados.')

# v4 usa sintetico_roturas_v2
copiar_dir(RUTA_SINTETICO,    'Datos/sintetico/roturas_v2')
copiar_dir(RUTA_FB_PROCESADO, 'Datos/fantastic_breaks/procesado')

print()
for c in ['Datos/sintetico/roturas_v2', 'Datos/fantastic_breaks/procesado']:
    n = len(list(Path(c).glob('*.npy'))) if Path(c).exists() else 0
    print(f'  {c}: {n} .npy')

In [ ]:
# ── CELDA 6: Verificar GPU ─────────────────────────────────────
import torch
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('Sin GPU. Ve a Entorno de ejecución → Cambiar tipo → A100 o T4')

In [ ]:
# ── CELDA 7: ENTRENAR v4 ───────────────────────────────────────
#
# Cambios respecto a v3:
#   --w_coarse 1.0  (antes 0.5) — igual peso para decoder coarse y fine
#   --epochs 500    (antes 400) — v3 convergió en 347, se da más margen
#   carpetas apuntan a roturas_v2
#
# El filtro de outliers ya está activo en dataset.py (FILTRAR_OUTLIERS=True)

!python -m E3.train \
    --carpetas   Datos/sintetico/roturas_v2 Datos/fantastic_breaks/procesado \
    --epochs     500 \
    --lr         1e-4 \
    --lr_decay   100 \
    --batch_size 64 \
    --w_coarse   1.0

In [ ]:
# ── CELDA 8: Guardar modelo en Drive ──────────────────────────
import shutil
from pathlib import Path

Path(RUTA_SALIDA_MODELO).mkdir(parents=True, exist_ok=True)
shutil.copy2('E3/checkpoints/best.pt', f'{RUTA_SALIDA_MODELO}/best.pt')
print(f'Modelo v4 guardado: {RUTA_SALIDA_MODELO}/best.pt')

for ckpt in sorted(Path('E3/checkpoints').glob('epoch_*.pt')):
    shutil.copy2(ckpt, Path(RUTA_SALIDA_MODELO) / ckpt.name)
    print(f'  + {ckpt.name}')

In [ ]:
# ── CELDA 9: Evaluar modelo v4 ─────────────────────────────────
# Compara con v3: CD=0.0665, F-Score=0.024

!python -m E3.evaluate \
    --checkpoint E3/checkpoints/best.pt \
    --carpetas   Datos/sintetico/roturas_v2 Datos/fantastic_breaks/procesado \
    --salida     E3/resultados

import shutil
from pathlib import Path

Path(RUTA_SALIDA_RESULTADOS).mkdir(parents=True, exist_ok=True)
shutil.copytree('E3/resultados', RUTA_SALIDA_RESULTADOS, dirs_exist_ok=True)
print(f'Resultados v4 guardados: {RUTA_SALIDA_RESULTADOS}')

In [ ]:
# ── CELDA 10: Comparar v3 vs v4 ────────────────────────────────
# Lee los resumen.txt de ambas versiones y los pone lado a lado.

from pathlib import Path

for version, ruta in [('v3', f'{BASE_E3}/resultados/v3_pcn/resumen.txt'),
                       ('v4', 'E3/resultados/resumen.txt')]:
    p = Path(ruta)
    print(f'=== {version} ===')
    if p.exists():
        print(p.read_text())
    else:
        print(f'  No encontrado: {ruta}')
    print()

In [ ]:
# ── CELDA 11: Ver figuras inline ────────────────────────────────
from IPython.display import Image, display
from pathlib import Path

figuras = sorted(Path('E3/resultados').glob('figura_*.png'))
if not figuras:
    print('No hay figuras. Ejecuta primero la Celda 9.')
else:
    for f in figuras:
        print(f.name)
        display(Image(str(f), width=1000))

In [ ]:
# ── CELDA 12: Visualización 3D interactiva ─────────────────────
#   Azul  = entrada rota
#   Verde = GT completa
#   Rojo  = predicción del modelo

import subprocess
subprocess.run(['pip', 'install', 'plotly', '--quiet'])

import plotly.graph_objects as go
import numpy as np
import torch
from E3.train import PCN, chamfer_distance
from E3.dataset import construir_dataloaders

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = PCN().to(device)
ckpt   = torch.load('E3/checkpoints/best.pt', map_location=device, weights_only=True)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f'Modelo cargado — época {ckpt["epoch"]}')

_, _, test_loader = construir_dataloaders(
    carpetas=['Datos/sintetico/roturas_v2', 'Datos/fantastic_breaks/procesado'],
    batch_size=32, augmentar=False,
)

rotos, gts, preds, cds = [], [], [], []
with torch.no_grad():
    for roto_b, gt_b in test_loader:
        _, pred_b = model(roto_b.to(device))
        pred_b = pred_b.cpu()
        for i in range(len(roto_b)):
            p, g = pred_b[i:i+1], gt_b[i:i+1]
            cd = chamfer_distance(p, g).item()
            rotos.append(roto_b[i].numpy())
            gts.append(gt_b[i].numpy())
            preds.append(pred_b[i].numpy())
            cds.append(cd)

cds_arr = np.array(cds)
orden   = np.argsort(cds_arr)
indices = list(orden[:3]) + list(orden[-3:])
titulos = ['Mejor 1', 'Mejor 2', 'Mejor 3', 'Peor 1', 'Peor 2', 'Peor 3']
print(f'CD — mejor: {cds_arr[orden[0]]:.4f} | peor: {cds_arr[orden[-1]]:.4f} | media: {cds_arr.mean():.4f}')

for idx, titulo in zip(indices, titulos):
    fig = go.Figure([
        go.Scatter3d(x=rotos[idx][:,0], y=rotos[idx][:,1], z=rotos[idx][:,2],
                     mode='markers', marker=dict(size=2, color='#4C72B0', opacity=0.55), name='Rota'),
        go.Scatter3d(x=gts[idx][:,0],   y=gts[idx][:,1],   z=gts[idx][:,2],
                     mode='markers', marker=dict(size=2, color='#55A868', opacity=0.35), name='GT'),
        go.Scatter3d(x=preds[idx][:,0], y=preds[idx][:,1], z=preds[idx][:,2],
                     mode='markers', marker=dict(size=2, color='#C44E52', opacity=0.85), name='Pred'),
    ])
    fig.update_layout(
        title=f'{titulo} — CD={cds[idx]:.4f}',
        scene=dict(xaxis=dict(range=[-1,1], showticklabels=False),
                   yaxis=dict(range=[-1,1], showticklabels=False),
                   zaxis=dict(range=[-1,1], showticklabels=False),
                   aspectmode='cube'),
        height=500, margin=dict(l=0,r=0,b=0,t=40)
    )
    fig.show()